In [7]:
import os
import random

import requests
from astrapy import DataAPIClient
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from github import Auth, Github
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

client = DataAPIClient(os.getenv("ASTRA_DB_APPLICATION_TOKEN"))
db = client.get_database(os.getenv("ASTRA_DB_API_ENDPOINT"))

gh = Github(auth=Auth.Token(os.getenv("GITHUB_PAT")))
repo = gh.get_repo("PrashantAghara/fastapi")

print("Setup complete")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3195.88it/s]


Setup complete


In [18]:
from astrapy.constants import VectorMetric
from astrapy.info import CollectionDefinition, CollectionVectorOptions

COLLECTION_NAME = "codeguardian_style_corpus"

if COLLECTION_NAME in db.list_collection_names():
    db.drop_collection(COLLECTION_NAME)
    print(f"Dropped existing collection: {COLLECTION_NAME}")

collection_definition = CollectionDefinition(
    vector=CollectionVectorOptions(
        dimension=384,
        metric=VectorMetric.COSINE,
    ),
)

collection = db.create_collection(COLLECTION_NAME, definition=collection_definition)
print(f"Created collection with vector search: {COLLECTION_NAME}")

Dropped existing collection: codeguardian_style_corpus
Created collection with vector search: codeguardian_style_corpus


In [19]:
guide_docs = []

try:
    content = repo.get_contents("docs/en/docs/contributing.md").decoded_content.decode("utf-8")
    guide_docs.append({"source": "docs/en/docs/contributing.md", "type": "guide", "content": content})
except Exception as e:
    print(f"Skipping repo contributing.md: {e}")

def fetch_external_guide(url: str) -> str:
    resp = requests.get(url, timeout=10)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")
    main = soup.find("main") or soup.find("article") or soup.body
    for tag in main.find_all(["script", "style", "nav", "footer"]):
        tag.decompose()
    return main.get_text(separator="\n", strip=True)

try:
    external_text = fetch_external_guide("https://tiangolo.com/open-source/contributing/")
    guide_docs.append({"source": "tiangolo.com/open-source/contributing", "type": "guide", "content": external_text})
except Exception as e:
    print(f"Skipping external guide: {e}")

print(f"Guide docs fetched: {len(guide_docs)}")
for d in guide_docs:
    print(f"  {d['source']} — {len(d['content'])} chars")

Guide docs fetched: 2
  docs/en/docs/contributing.md — 271 chars
  tiangolo.com/open-source/contributing — 4137 chars


In [20]:
contents = repo.get_contents("fastapi")
py_files = [f for f in contents if f.name.endswith(".py")]
sample = random.sample(py_files, min(20, len(py_files)))

code_docs = []
for f in sample:
    content = f.decoded_content.decode("utf-8")
    code_docs.append({"source": f.path, "type": "code_sample", "content": content})

print(f"Code samples fetched: {len(code_docs)}")

Code samples fetched: 20


In [21]:
all_docs = guide_docs + code_docs
records = []

for doc in all_docs:
    chunks = splitter.split_text(doc["content"])
    for i, chunk in enumerate(chunks):
        vector = embeddings.embed_query(chunk)
        records.append({
            "$vector": vector,
            "text": chunk,
            "source": doc["source"],
            "type": doc["type"],
            "chunk_index": i,
        })

collection.insert_many(records)
print(f"Ingested {len(records)} chunks from {len(all_docs)} documents")

Ingested 1086 chunks from 22 documents


In [22]:
query = "type hints and function naming conventions"
query_vector = embeddings.embed_query(query)

results = collection.find(sort={"$vector": query_vector}, limit=5)
for r in results:
    print(f"[{r['type']}] {r['source']} (chunk {r['chunk_index']})")
    print(r["text"][:200])
    print("---")

[code_sample] fastapi/routing.py (chunk 165)
@property
    def name(self) -> str | None:
        return getattr(self._effective_route, "name", None)

    @property
    def methods(self) -> set[str] | None:
        return getattr(self._effective_
---
[code_sample] fastapi/routing.py (chunk 654)
It could be any valid Pydantic *field* type. So, it doesn't have to
                be a Pydantic model, it could be other things, like a `list`, `dict`,
                etc.

                It will 
---
[code_sample] fastapi/routing.py (chunk 525)
It could be any valid Pydantic *field* type. So, it doesn't have to
                be a Pydantic model, it could be other things, like a `list`, `dict`,
                etc.

                It will 
---
[code_sample] fastapi/routing.py (chunk 353)
It could be any valid Pydantic *field* type. So, it doesn't have to
                be a Pydantic model, it could be other things, like a `list`, `dict`,
                etc.

                It will 
---


In [ ]:
style_prompt_rag = """You are a Style Agent reviewing a pull request diff.

Below is retrieved context from the project's contributing guide and existing codebase, which reflects its actual conventions:

{retrieved_context}

Diff:
{diff}

Based ONLY on patterns and conventions evident in the retrieved context above — not general best practices —
identify style issues in the diff. For each: filename, line (from the diff hunk header context), a short comment, and severity (info/warning).
If the retrieved context doesn't clearly support a judgment, say so rather than guessing.
If no issues, say so explicitly. End with a one-line verdict: PASS, PASS_WITH_WARNINGS, or FAIL."""

def run_style_agent_rag(pr, filenames: list[str]) -> str:
    diff_text = get_pr_diff_text(pr, filenames)
    retrieved_context = retrieve_style_context(diff_text)
    result = llm.invoke(
        style_prompt_rag.format(retrieved_context=retrieved_context, diff=diff_text)
    )
    return result.content